# 09 - Desigualdad territorial ENIGH 2018-2024

Este notebook resume una etapa descriptiva de desigualdad territorial para la tesina. El objetivo es calcular Gini ponderado, compararlo con el benchmark de Banco de México cuando la definición sea compatible, y separar desigualdad dentro de territorios de brechas entre territorios.

No se crean modelos ni se deflacta. Se incorpora solo una fuente oficial acotada para zonas metropolitanas y no se aproximan grandes urbes con municipios sueltos.

## Definición metodológica

Benchmark revisado: Banco de México, Recuadro 2 del Reporte sobre las Economías Regionales enero-marzo 2024, "Disminución de la desigualdad de ingresos regional en un contexto de crecimiento propobre: 2018-2022".

Punto clave: el recuadro indica que el Gini usa **ingreso corriente total promedio por hogar**. Por eso la comparación principal de este notebook usa:

- ingreso: `ing_cor_hogar_oficial_tri`;
- ponderador: `factor`;
- universo: todos los hogares con ingreso no faltante, no negativo y factor positivo;
- ceros: se conservan como valores válidos;
- escala: se reporta Gini en 0-1 y 0-100;
- años comparables con Banxico: 2018, 2020 y 2022.

La comparación no busca calzar perfectamente. Banco de México usa bases generadas por CONEVAL a partir de ENIGH; aquí se usa el mart propio con la variable oficial de `concentradohogar`. Diferencias pequeñas o moderadas pueden venir de definición CONEVAL, procesamiento, universo, ponderación, escala temporal del ingreso o redondeo.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlesize": 13, "axes.labelsize": 11})

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "interim" / "revision_4").exists():
            return candidate
    raise FileNotFoundError("No se encontró data/interim/revision_4 desde el directorio actual.")

ROOT = find_project_root()
REV4 = ROOT / "data" / "interim" / "revision_4"
HOGAR_PATH = REV4 / "mart_hogar_2018_2024.csv.gz"
PERSONA_PATH = REV4 / "mart_persona_2018_2024.csv.gz"

WEIGHT = "factor"
INC_HH = "ing_cor_hogar_oficial_tri"
INC_PC = "ing_cor_pc_oficial_tri"
INC_LAB = "ingreso_persona_laboral_negocio_tri"
YEARS = [2018, 2020, 2022, 2024]

print("Rutas relativas detectadas:")
print(f"Base hogares: {HOGAR_PATH.exists()} | {HOGAR_PATH.relative_to(ROOT).as_posix()}")
print(f"Base personas: {PERSONA_PATH.exists()} | {PERSONA_PATH.relative_to(ROOT).as_posix()}")

## Figuras documentales y llaves

Las figuras que se insertan en la documentación se guardan en `reports/figures_documentacion/`, una carpeta específica y versionable. No se usan rutas absolutas en el notebook ni en el Markdown.


In [ ]:
RAW = ROOT / "data" / "raw" / "EINGH"
DOC_FIG_DIR = ROOT / "reports" / "figures_documentacion"
DOC_FIG_DIR.mkdir(parents=True, exist_ok=True)
ZM_MAP_PATH = ROOT / "docs" / "zonas_metropolitanas_prioritarias_2020.csv"


def save_doc_figure(fig, filename, alt_text):
    """Guarda figura versionable y la muestra en el notebook por ruta relativa."""
    path = DOC_FIG_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Markdown(f"![{alt_text}](../reports/figures_documentacion/{filename})"))
    return path


def as_code(series, width):
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype("string").str.zfill(width)


## Carga de bases analíticas

Se cargan únicamente las columnas necesarias para esta revisión. La base de hogares se usa para Gini comparable con Banxico; la base de personas se usa solo para ingreso laboral individual en la sección CDMX.

In [ ]:
hogar_cols = [
    "anio", "region_banxico", "entidad", "cve_ent", "tam_loc_desc", "est_socio_desc",
    WEIGHT, "factor_hogar", "est_dis", "upm", INC_HH, INC_PC, "ingtrab_hogar_oficial_tri", "tot_integ",
]
persona_cols = [
    "anio", "region_banxico", "entidad", "cve_ent", "tam_loc_desc", "est_socio_desc",
    "sexo_desc", WEIGHT, INC_LAB, "ing_cor_hogar_pc_oficial_tri", "edad",
]
hogar = pd.read_csv(HOGAR_PATH, usecols=hogar_cols, low_memory=False)
persona = pd.read_csv(PERSONA_PATH, usecols=persona_cols, low_memory=False)

for col in ["anio", WEIGHT, "factor_hogar", INC_HH, INC_PC, "ingtrab_hogar_oficial_tri", "tot_integ"]:
    hogar[col] = pd.to_numeric(hogar[col], errors="coerce")
for col in ["anio", WEIGHT, INC_LAB, "ing_cor_hogar_pc_oficial_tri", "edad"]:
    persona[col] = pd.to_numeric(persona[col], errors="coerce")

hogar["peso_persona_expandida"] = hogar[WEIGHT] * hogar["tot_integ"]

validacion_base = pd.DataFrame([
    {"base": "hogar", "filas": len(hogar), "columnas": hogar.shape[1], "factor_missing": int(hogar[WEIGHT].isna().sum()), "ingreso_missing": int(hogar[INC_HH].isna().sum()), "ingreso_negativo": int(hogar[INC_HH].lt(0).sum())},
    {"base": "persona", "filas": len(persona), "columnas": persona.shape[1], "factor_missing": int(persona[WEIGHT].isna().sum()), "ingreso_laboral_missing": int(persona[INC_LAB].isna().sum()), "ingreso_laboral_negativo": int(persona[INC_LAB].lt(0).sum())},
])
display(validacion_base)

## Funciones ponderadas

La fórmula operativa del Gini usa la curva de Lorenz ponderada. Se ordena el ingreso de menor a mayor, se acumula población ponderada e ingreso ponderado, y se calcula:

```text
Gini = 1 - 2 * área bajo la curva de Lorenz
```

El cálculo conserva ceros y excluye únicamente ingresos faltantes, negativos o pesos no positivos.

In [ ]:
def weighted_quantile(values, weights, qs):
    x = pd.to_numeric(values, errors="coerce").to_numpy(dtype="float64")
    w = pd.to_numeric(weights, errors="coerce").to_numpy(dtype="float64")
    q = np.atleast_1d(qs).astype(float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[mask], w[mask]
    if len(x) == 0:
        return np.full(len(q), np.nan)
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum_w = np.cumsum(w) / np.sum(w)
    return np.interp(q, cum_w, x, left=x[0], right=x[-1])

def weighted_gini(values, weights):
    x = pd.to_numeric(values, errors="coerce").to_numpy(dtype="float64")
    w = pd.to_numeric(weights, errors="coerce").to_numpy(dtype="float64")
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0) & (x >= 0)
    x, w = x[mask], w[mask]
    if len(x) == 0 or np.sum(x * w) <= 0:
        return np.nan
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum_w = np.cumsum(w)
    cum_xw = np.cumsum(x * w)
    pop_share = np.insert(cum_w / cum_w[-1], 0, 0)
    income_share = np.insert(cum_xw / cum_xw[-1], 0, 0)
    return float(1 - 2 * np.trapezoid(income_share, pop_share))

def weighted_summary(df, group_cols, income_col, positive_only=False):
    work = df[df[income_col].gt(0)].copy() if positive_only else df.copy()
    rows = []
    for key, g in work.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        x = pd.to_numeric(g[income_col], errors="coerce")
        w = pd.to_numeric(g[WEIGHT], errors="coerce")
        mask = x.notna() & w.notna() & w.gt(0) & x.ge(0)
        x, w = x[mask], w[mask]
        qs = weighted_quantile(x, w, [0.25, 0.50, 0.75, 0.90, 0.95])
        row.update({
            "n": int(mask.sum()),
            "n_ponderado": float(w.sum()),
            "media": float(np.average(x, weights=w)) if len(x) else np.nan,
            "p25": qs[0],
            "mediana": qs[1],
            "p75": qs[2],
            "p90": qs[3],
            "p95": qs[4],
            "gini": weighted_gini(x, w),
        })
        rows.append(row)
    out = pd.DataFrame(rows)
    out["gini_0_100"] = out["gini"] * 100
    return out

def money_cols(df):
    cols = ["media", "p25", "mediana", "p75", "p90", "p95", "gini", "gini_0_100"]
    return df.round({c: 2 for c in cols if c in df.columns})

## Auditoría de `factor`

En las bases analíticas, `factor` es alias de `factor_hogar`. La metadata ENIGH del proyecto lo define como factor de expansión y lo ubica junto a `est_dis` y `upm`. Para hogares se usa directamente; para personas cada integrante hereda el factor del hogar; para ingreso corriente per cápita como distribución de bienestar individual se usa `factor * tot_integ`.


In [ ]:
metadata = pd.read_csv(ROOT / "docs" / "enigh_variable_metadata.csv")
year_col = "anio" if "anio" in metadata.columns else "year"
table_col = "tabla" if "tabla" in metadata.columns else "table"
factor_metadata = metadata[metadata["variable"].eq("factor")].copy()
factor_disponibilidad = (
    factor_metadata.assign(disponible="Sí")
    .pivot_table(index=table_col, columns=year_col, values="disponible", aggfunc="first", fill_value="No")
    .reset_index()
    .rename(columns={table_col: "tabla"})
)
display(factor_disponibilidad)

factor_resumen = hogar.groupby("anio").agg(
    n_hogares=(WEIGHT, "size"),
    hogares_expandidos=(WEIGHT, "sum"),
    poblacion_expandida=("tot_integ", lambda s: float((s * hogar.loc[s.index, WEIGHT]).sum())),
    factor_min=(WEIGHT, "min"),
    factor_mediana=(WEIGHT, "median"),
    factor_max=(WEIGHT, "max"),
).reset_index()
display(factor_resumen.round(2))


### Contraste de `factor` contra tablas originales

Cuando `factor` existe en las tablas raw, se contrasta contra `concentradohogar` por las llaves disponibles. En 2018 y 2020 varias tablas hijas no lo contienen directamente; en 2022 y 2024 sí aparece repetido y coincide con el factor del hogar.


In [ ]:
def audit_factor_across_raw_tables(raw_root, years):
    tables = ["viviendas", "hogares", "poblacion", "trabajos", "ingresos", "gastoshogar", "gastospersona", "concentradohogar"]
    rows = []
    for year in years:
        conc = pd.read_csv(raw_root / str(year) / "concentradohogar.csv", usecols=["folioviv", "foliohog", "factor"], low_memory=False)
        conc["folioviv"] = conc["folioviv"].astype(str)
        conc["foliohog"] = conc["foliohog"].astype(str)
        conc["factor"] = pd.to_numeric(conc["factor"], errors="coerce")
        for table in tables:
            path = raw_root / str(year) / f"{table}.csv"
            cols = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
            if "factor" not in cols:
                rows.append({"anio": year, "tabla": table, "factor_en_tabla": "No", "n_filas": np.nan, "mismatches_con_concentrado": np.nan, "max_dif_factor": np.nan})
                continue
            usecols = ["folioviv", "factor"] + (["foliohog"] if "foliohog" in cols else [])
            df = pd.read_csv(path, usecols=usecols, low_memory=False)
            df["folioviv"] = df["folioviv"].astype(str)
            df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
            if "foliohog" in df.columns:
                df["foliohog"] = df["foliohog"].astype(str)
                merged = df.merge(conc, on=["folioviv", "foliohog"], how="left", suffixes=("_tabla", "_concentrado"))
            else:
                merged = df.merge(conc, on="folioviv", how="left", suffixes=("_tabla", "_concentrado"))
            diff = (merged["factor_tabla"] - merged["factor_concentrado"]).abs()
            rows.append({"anio": year, "tabla": table, "factor_en_tabla": "Sí", "n_filas": len(df), "mismatches_con_concentrado": int((diff.fillna(0) > 0).sum()), "max_dif_factor": float(diff.max()) if diff.notna().any() else np.nan})
    return pd.DataFrame(rows)

factor_tablas = audit_factor_across_raw_tables(RAW, YEARS)
display(factor_tablas)
display(factor_tablas.groupby(["anio", "factor_en_tabla"], dropna=False).agg(tablas=("tabla", "count"), mismatches=("mismatches_con_concentrado", "sum")).reset_index())


## Tabla compacta de auditoría de ponderaciones

Esta tabla resume cómo se deben leer los principales resultados del proyecto. Evita vender como poblacional un resultado muestral.


In [ ]:
auditoria_ponderaciones = pd.DataFrame([
    {"Resultado": "Gini nacional y regional", "Nivel": "Hogar", "Variable": INC_HH, "Ponderado": "Sí", "Peso utilizado": "factor", "Observación": "Estimación puntual ponderada de hogares; ceros conservados."},
    {"Resultado": "Media, mediana y cuantiles de ingreso corriente del hogar", "Nivel": "Hogar", "Variable": INC_HH, "Ponderado": "Sí", "Peso utilizado": "factor", "Observación": "Pesos nominales trimestrales; no deflactado."},
    {"Resultado": "Ingreso corriente per cápita territorial", "Nivel": "Personas en hogares", "Variable": INC_PC, "Ponderado": "Sí", "Peso utilizado": "factor * tot_integ", "Observación": "Cada integrante hereda el ingreso per cápita de su hogar."},
    {"Resultado": "Ingreso laboral individual positivo", "Nivel": "Persona", "Variable": INC_LAB, "Ponderado": "Sí", "Peso utilizado": "factor", "Observación": "Condicionado a ingreso laboral positivo."},
    {"Resultado": "Hallazgos preliminares del notebook 07", "Nivel": "Persona/Hogar", "Variable": "ingreso laboral y per cápita", "Ponderado": "No", "Peso utilizado": "Ninguno", "Observación": "La función summary_table reportaba medianas/cuartiles muestrales."},
    {"Resultado": "Calidad y faltantes del notebook 08", "Nivel": "Variable", "Variable": "faltantes, ceros, Cramér's V, SMD", "Ponderado": "No", "Peso utilizado": "Ninguno", "Observación": "Auditoría de datos, no estimación poblacional."},
])
display(auditoria_ponderaciones)


## Benchmark Banxico 2018-2022

Los valores de Banxico se capturan tal como aparecen en la tabla oficial, en escala 0-100.

In [ ]:
banxico_gini = pd.DataFrame([
    ("Norte", 2018, 43.1), ("Norte", 2020, 43.7), ("Norte", 2022, 40.3),
    ("Centro Norte", 2018, 43.2), ("Centro Norte", 2020, 41.3), ("Centro Norte", 2022, 40.4),
    ("Centro", 2018, 45.1), ("Centro", 2020, 44.5), ("Centro", 2022, 42.1),
    ("Sur", 2018, 47.5), ("Sur", 2020, 45.8), ("Sur", 2022, 44.9),
    ("Nacional", 2018, 45.7), ("Nacional", 2020, 45.0), ("Nacional", 2022, 43.1),
], columns=["region_banxico", "anio", "gini_banxico_0_100"])
display(banxico_gini)

## Gini nacional

Este es el cálculo principal con `ing_cor_hogar_oficial_tri`. La lectura temporal debe hacerse con cautela porque los ingresos están en pesos nominales; el Gini dentro de cada año no depende de multiplicar todos los ingresos por una constante, pero la comparación temporal sustantiva sí requiere más contexto.

In [ ]:
gini_nacional = weighted_summary(hogar, ["anio"], INC_HH)
display(money_cols(gini_nacional[["anio", "n", "n_ponderado", "media", "mediana", "gini", "gini_0_100"]]))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.lineplot(data=gini_nacional, x="anio", y="gini_0_100", marker="o", ax=ax)
ax.set_title("Gini nacional ponderado: ingreso corriente total del hogar")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks(YEARS)
display(fig)
plt.close(fig)

## Gini por región Banxico y validación

La validación compara dirección, orden relativo y magnitud aproximada. No se fuerza el resultado a coincidir con Banxico.

In [ ]:
gini_region = weighted_summary(hogar, ["anio", "region_banxico"], INC_HH)
gini_region_display = money_cols(gini_region[["anio", "region_banxico", "n", "media", "mediana", "gini_0_100"]].sort_values(["anio", "region_banxico"]))
display(gini_region_display)

propio_para_banxico = pd.concat([
    gini_region[["anio", "region_banxico", "gini_0_100"]],
    gini_nacional.assign(region_banxico="Nacional")[["anio", "region_banxico", "gini_0_100"]],
], ignore_index=True)
comparacion_banxico = propio_para_banxico.merge(banxico_gini, on=["anio", "region_banxico"], how="inner")
comparacion_banxico["dif_puntos"] = comparacion_banxico["gini_0_100"] - comparacion_banxico["gini_banxico_0_100"]
comparacion_banxico["abs_dif_puntos"] = comparacion_banxico["dif_puntos"].abs()
display(money_cols(comparacion_banxico.sort_values(["anio", "region_banxico"])))

discrepancia_max = comparacion_banxico.loc[comparacion_banxico["abs_dif_puntos"].idxmax()].copy()
display(Markdown(
    f"**Mayor discrepancia:** {discrepancia_max['region_banxico']} {int(discrepancia_max['anio'])}, "
    f"{discrepancia_max['dif_puntos']:.2f} puntos frente a Banxico."
))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=gini_region, x="anio", y="gini_0_100", hue="region_banxico", marker="o", ax=ax)
ax.set_title("Gini por región Banxico (Gini ponderado)")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks(YEARS)
ax.legend(title="Región")
save_doc_figure(fig, "gini_regiones_2018_2024.png", "Evolución del Gini por región Banxico")

## Auditoría de reproducción del Gini de Banco de México

Primero se calcula el Gini directamente desde `concentradohogar.csv` con la misma variable (`ing_cor`), el mismo factor y el mismo universo. Si coincide con el mart, la discrepancia con Banxico no viene de la construcción de la base analítica.


In [ ]:
def read_concentradohogar_raw(year):
    raw = pd.read_csv(RAW / str(year) / "concentradohogar.csv", usecols=["folioviv", "foliohog", "ubica_geo", "factor", "ing_cor", "tot_integ"], low_memory=False)
    raw["anio"] = year
    raw["folioviv"] = raw["folioviv"].astype(str)
    raw["foliohog"] = raw["foliohog"].astype(str)
    raw["factor"] = pd.to_numeric(raw["factor"], errors="coerce")
    raw["ing_cor"] = pd.to_numeric(raw["ing_cor"], errors="coerce")
    raw["tot_integ"] = pd.to_numeric(raw["tot_integ"], errors="coerce")
    return raw

raw_concentrado = pd.concat([read_concentradohogar_raw(year) for year in YEARS], ignore_index=True)
gini_raw = weighted_summary(raw_concentrado, ["anio"], "ing_cor").rename(columns={"gini_0_100": "gini_concentradohogar_0_100"})
mart_vs_raw = gini_nacional[["anio", "gini_0_100"]].merge(gini_raw[["anio", "gini_concentradohogar_0_100"]], on="anio")
mart_vs_raw["diferencia_pp"] = mart_vs_raw["gini_0_100"] - mart_vs_raw["gini_concentradohogar_0_100"]
display(mart_vs_raw.round(4))

checks = []
for year in YEARS:
    raw_year = raw_concentrado[raw_concentrado["anio"].eq(year)]
    mart_year = hogar[hogar["anio"].eq(year)][["folioviv", "foliohog", "factor", INC_HH]] if {"folioviv", "foliohog"}.issubset(hogar.columns) else pd.read_csv(HOGAR_PATH, usecols=["anio", "folioviv", "foliohog", "factor", INC_HH], low_memory=False).query("anio == @year")
    mart_year["folioviv"] = mart_year["folioviv"].astype(str)
    mart_year["foliohog"] = mart_year["foliohog"].astype(str)
    merged = raw_year.merge(mart_year, on=["folioviv", "foliohog"], how="left", suffixes=("_raw", "_mart"))
    checks.append({"anio": year, "filas_raw": len(raw_year), "filas_mergeadas": int(merged[INC_HH].notna().sum()), "max_abs_dif_ingreso": float((merged["ing_cor"] - merged[INC_HH]).abs().max()), "max_abs_dif_factor": float((merged["factor_raw"] - merged["factor_mart"]).abs().max())})
mart_raw_check = pd.DataFrame(checks)
display(mart_raw_check.round(4))


## Ponderado vs no ponderado y diagnóstico de definición

El cambio de ponderar no explica por sí solo la distancia frente a Banxico. Además de la definición hogar-total, se calcula una variante diagnóstica con `ing_cor_pc_oficial_tri` y `factor * tot_integ`, porque se acerca mucho más al benchmark. Esta variante ayuda a diagnosticar, pero no sustituye la definición oficial hasta reproducir exactamente las bases CONEVAL usadas por Banxico.


In [ ]:
def weighted_summary_custom(df, group_cols, income_col, weight_col, positive_only=False):
    work = df[df[income_col].gt(0)].copy() if positive_only else df.copy()
    rows = []
    for key, group in work.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        x = pd.to_numeric(group[income_col], errors="coerce")
        w = pd.to_numeric(group[weight_col], errors="coerce")
        mask = x.notna() & w.notna() & w.gt(0) & x.ge(0)
        x, w = x[mask], w[mask]
        qs = weighted_quantile(x, w, [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        row.update({"n_muestral": int(mask.sum()), "poblacion_expandida": float(w.sum()), "media_ponderada": float(np.average(x, weights=w)) if len(x) else np.nan, "p10": qs[0], "p25": qs[1], "mediana_ponderada": qs[2], "p75": qs[3], "p90": qs[4], "p95": qs[5], "p99": qs[6], "maximo": float(np.nanmax(x)) if len(x) else np.nan, "gini_ponderado_0_100": weighted_gini(x, w) * 100})
        rows.append(row)
    return pd.DataFrame(rows)

weighted_unweighted = []
for year, group in hogar[hogar["anio"].isin([2018, 2020, 2022])].groupby("anio"):
    x = pd.to_numeric(group[INC_HH], errors="coerce")
    w_unweighted = pd.Series(np.ones(len(group)), index=group.index)
    weighted_unweighted.append({"anio": year, "gini_ponderado_0_100": weighted_gini(x, group["factor"]) * 100, "gini_no_ponderado_0_100": weighted_gini(x, w_unweighted) * 100})
weighted_unweighted = pd.DataFrame(weighted_unweighted).merge(banxico_gini[banxico_gini["region_banxico"].eq("Nacional")][["anio", "gini_banxico_0_100"]], on="anio")
display(weighted_unweighted.round(2))

variant_rows = []
for year, group in hogar.groupby("anio"):
    variant_rows.append({"region_banxico": "Nacional", "anio": year, "gini_hogar_total_factor": weighted_gini(group[INC_HH], group["factor"]) * 100, "gini_pc_peso_persona": weighted_gini(group[INC_PC], group["peso_persona_expandida"]) * 100})
for (year, region), group in hogar.groupby(["anio", "region_banxico"]):
    variant_rows.append({"region_banxico": region, "anio": year, "gini_hogar_total_factor": weighted_gini(group[INC_HH], group["factor"]) * 100, "gini_pc_peso_persona": weighted_gini(group[INC_PC], group["peso_persona_expandida"]) * 100})

diag_banxico = pd.DataFrame(variant_rows).merge(banxico_gini, on=["region_banxico", "anio"], how="inner")
diag_banxico["dif_hogar_total_pp"] = diag_banxico["gini_hogar_total_factor"] - diag_banxico["gini_banxico_0_100"]
diag_banxico["dif_pc_persona_pp"] = diag_banxico["gini_pc_peso_persona"] - diag_banxico["gini_banxico_0_100"]
display(diag_banxico.sort_values(["anio", "region_banxico"]).round(2))

fig, ax = plt.subplots(figsize=(9, 5))
nacional_diag = diag_banxico[diag_banxico["region_banxico"].eq("Nacional")].sort_values("anio")
ax.plot(nacional_diag["anio"], nacional_diag["gini_banxico_0_100"], "s--", linewidth=2.2, label="Banxico")
ax.plot(nacional_diag["anio"], nacional_diag["gini_hogar_total_factor"], "o-", linewidth=2.2, label="Mart: hogar total ponderado")
ax.plot(nacional_diag["anio"], nacional_diag["gini_pc_peso_persona"], "^-", linewidth=2.2, label="Diagnóstico: per cápita ponderado por población")
ax.set_title("Gini nacional propio vs Banxico")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks([2018, 2020, 2022])
ax.legend()
ax.text(0.01, -0.20, "La variante per cápita es diagnóstica; no reemplaza la definición primaria sin validar CONEVAL.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "gini_banxico_comparacion_2018_2022.png", "Gini nacional propio vs Banxico")


## Cola de la distribución

Se revisan percentiles ponderados y máximos por año. No se eliminan outliers ni se winsoriza; la meta es diagnosticar si la diferencia con Banxico parece venir de colas, universo o definición de ingreso.


In [ ]:
colas_ingreso = weighted_summary_custom(hogar, ["anio"], INC_HH, "factor")[["anio", "p10", "p25", "mediana_ponderada", "p75", "p90", "p95", "p99", "maximo"]]
display(colas_ingreso.round(2))


## Tabla final de validación Banxico

La conclusión se clasifica como reproducción cercana cuando la diferencia absoluta es menor o igual a 1 punto de Gini; en caso contrario se conserva como reproducción parcial.


In [ ]:
comparacion_banxico_final = diag_banxico[["anio", "region_banxico", "gini_hogar_total_factor", "gini_banxico_0_100", "dif_hogar_total_pp"]].copy()
comparacion_banxico_final = comparacion_banxico_final.rename(columns={"gini_hogar_total_factor": "nuestro_gini_0_100", "gini_banxico_0_100": "banxico_0_100", "dif_hogar_total_pp": "diferencia_absoluta_pp"})
comparacion_banxico_final["diferencia_relativa_pct"] = 100 * comparacion_banxico_final["diferencia_absoluta_pp"] / comparacion_banxico_final["banxico_0_100"]
comparacion_banxico_final["conclusion"] = np.where(comparacion_banxico_final["diferencia_absoluta_pp"].abs().le(1), "reproducción cercana", "reproducción parcial")
display(comparacion_banxico_final.sort_values(["anio", "region_banxico"]).round(2))


## Gini por entidad, tamaño de localidad y estrato socioeconómico

Estas tablas miden desigualdad dentro de cada territorio o estrato. No son brechas entre territorios; por ejemplo, un Gini alto en una entidad indica mayor dispersión interna de ingresos de hogares dentro de esa entidad.

In [ ]:
gini_entidad    = weighted_summary(hogar, ["anio", "entidad"], INC_HH)
gini_tam_loc    = weighted_summary(hogar, ["anio", "tam_loc_desc"], INC_HH)
gini_est_socio  = weighted_summary(hogar, ["anio", "est_socio_desc"], INC_HH)

entidad_2024 = gini_entidad[gini_entidad["anio"].eq(2024)].sort_values("gini_0_100")
display(Markdown("### Entidades con menor Gini interno en 2024"))
display(money_cols(entidad_2024.head(8)[["entidad", "n", "mediana", "gini_0_100"]]))
display(Markdown("### Entidades con mayor Gini interno en 2024"))
display(money_cols(entidad_2024.tail(8)[["entidad", "n", "mediana", "gini_0_100"]]))

display(Markdown("### Tamaño de localidad, 2024"))
display(money_cols(gini_tam_loc[gini_tam_loc["anio"].eq(2024)][["tam_loc_desc", "n", "media", "mediana", "gini_0_100"]].sort_values("mediana", ascending=False)))

display(Markdown("### Estrato socioeconómico, 2024"))
display(money_cols(gini_est_socio[gini_est_socio["anio"].eq(2024)][["est_socio_desc", "n", "media", "mediana", "gini_0_100"]].sort_values("mediana", ascending=False)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=gini_tam_loc, x="anio", y="mediana", hue="tam_loc_desc", marker="o", ax=axes[0])
axes[0].set_title("Mediana ponderada por tamaño de localidad")
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Ingreso corriente hogar")
axes[0].legend(title="Tamaño", fontsize=8)
sns.lineplot(data=gini_est_socio, x="anio", y="mediana", hue="est_socio_desc", marker="o", ax=axes[1])
axes[1].set_title("Mediana ponderada por estrato socioeconómico")
axes[1].set_xlabel("Año")
axes[1].set_ylabel("Ingreso corriente hogar")
axes[1].legend(title="Estrato", fontsize=8)
fig.tight_layout()
display(fig)
plt.close(fig)

## CDMX

Para CDMX se calculan métricas descriptivas ponderadas por año. Los ingresos están en pesos nominales trimestrales. Para ingreso laboral individual se usa el universo con ingreso laboral positivo, porque la pregunta aquí es el monto entre quienes reportan ingreso laboral; la probabilidad de tener ingreso laboral positivo debe analizarse aparte.

In [ ]:
def cve_ent_2(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(2)

cdmx_hogar = hogar[cve_ent_2(hogar["cve_ent"]).eq("09")].copy()
cdmx_persona = persona[cve_ent_2(persona["cve_ent"]).eq("09")].copy()

cdmx_ingreso_hogar = weighted_summary(cdmx_hogar, ["anio"], INC_HH).assign(metrica="Ingreso corriente hogar")
cdmx_ingreso_pc = weighted_summary(cdmx_hogar, ["anio"], INC_PC).assign(metrica="Ingreso corriente per capita hogar")
cdmx_laboral = weighted_summary(cdmx_persona, ["anio"], INC_LAB, positive_only=True).assign(metrica="Ingreso laboral individual positivo")
cdmx_metricas = pd.concat([cdmx_ingreso_hogar, cdmx_ingreso_pc, cdmx_laboral], ignore_index=True)

display(money_cols(cdmx_metricas[["anio", "metrica", "n", "n_ponderado", "media", "p25", "mediana", "p75", "p90", "p95", "gini_0_100"]]))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=cdmx_metricas, x="anio", y="mediana", hue="metrica", marker="o", ax=ax)
ax.set_title("CDMX: medianas ponderadas por métrica")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos nominales trimestrales")
ax.set_xticks(YEARS)
ax.legend(title="Métrica", fontsize=8)
display(fig)
plt.close(fig)

## Brechas territoriales

Aquí se separa la idea de desigualdad interna y brecha territorial:

- desigualdad interna: Gini dentro de cada región, tamaño de localidad o estrato;
- brecha territorial: diferencia o razón entre medianas de grupos territoriales dentro del mismo año.

Esta no es una descomposición formal del Gini; es una lectura descriptiva para orientar la tesina.

In [ ]:
def median_gap(summary_df, dim):
    rows = []
    for year, g in summary_df.groupby("anio"):
        hi = g.loc[g["mediana"].idxmax()]
        lo = g.loc[g["mediana"].idxmin()]
        rows.append({
            "anio": int(year),
            "dimension": dim,
            "grupo_mayor_mediana": hi[dim],
            "mediana_mayor": hi["mediana"],
            "grupo_menor_mediana": lo[dim],
            "mediana_menor": lo["mediana"],
            "razon_mediana": hi["mediana"] / lo["mediana"],
            "brecha_mediana": hi["mediana"] - lo["mediana"],
        })
    return pd.DataFrame(rows)

brechas = pd.concat([
    median_gap(gini_region, "region_banxico"),
    median_gap(gini_tam_loc, "tam_loc_desc"),
    median_gap(gini_est_socio, "est_socio_desc"),
], ignore_index=True)
display(money_cols(brechas.sort_values(["dimension", "anio"])))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=brechas, x="anio", y="razon_mediana", hue="dimension", marker="o", ax=ax)
ax.set_title("Brecha territorial: razón entre mayor y menor mediana")
ax.set_xlabel("Año")
ax.set_ylabel("Razón de medianas")
ax.set_xticks(YEARS)
ax.legend(title="Dimensión")
display(fig)
plt.close(fig)

## Delimitación oficial de zonas metropolitanas

No se comparan municipios sueltos. Se usa `docs/zonas_metropolitanas_prioritarias_2020.csv`, un extracto versionable de **Las metrópolis de México 2020** de CONAPO/SEDATU/INEGI. La fuente etiqueta la zona como “Ciudad de México”; aquí se reporta como “Valle de México” para mantener el término de trabajo, conservando `nombre_oficial` en el mapping.


In [ ]:
zonas_prioritarias = ["Valle de México", "Guadalajara", "Monterrey"]
mapa_zm = pd.read_csv(ZM_MAP_PATH, dtype={"cve_ent": "string", "cve_mun": "string", "clave_compuesta_municipio": "string"})
mapa_zm["cve_ent"] = mapa_zm["cve_ent"].str.zfill(2)
mapa_zm["cve_mun"] = mapa_zm["cve_mun"].str.zfill(3)

resumen_mapa_zm = mapa_zm.groupby(["zona_metropolitana", "nombre_oficial", "tipo", "anio_delimitacion"]).agg(
    municipios_oficiales=("clave_compuesta_municipio", "nunique"),
    entidades=("entidad", lambda s: ", ".join(sorted(s.dropna().unique()))),
).reset_index()
display(resumen_mapa_zm)

display(mapa_zm[["zona_metropolitana", "cve_ent", "entidad", "cve_mun", "municipio"]].head(12))


## Cobertura metropolitana en ENIGH

Se valida cobertura por zona y año antes de interpretar: hogares, personas, población expandida, municipios presentes y combinaciones `est_dis + upm`. La lectura es descriptiva exploratoria, no inferencia formal con diseño muestral complejo.


In [ ]:
hogar_geo_cols = ["anio", "folioviv", "foliohog", "cve_ent", "cve_mun", "entidad", "municipio", "region_banxico", "tam_loc_desc", "est_socio_desc", "est_dis", "upm", "factor", "tot_integ", INC_HH, INC_PC]
persona_geo_cols = ["anio", "folioviv", "foliohog", "numren", "cve_ent", "cve_mun", "entidad", "municipio", "factor", INC_LAB]
hogar_geo = pd.read_csv(HOGAR_PATH, usecols=hogar_geo_cols, low_memory=False)
persona_geo = pd.read_csv(PERSONA_PATH, usecols=persona_geo_cols, low_memory=False)

for df in [hogar_geo, persona_geo]:
    df["cve_ent"] = as_code(df["cve_ent"], 2)
    df["cve_mun"] = as_code(df["cve_mun"], 3)
for col in ["anio", "factor", "tot_integ", INC_HH, INC_PC]:
    if col in hogar_geo.columns:
        hogar_geo[col] = pd.to_numeric(hogar_geo[col], errors="coerce")
for col in ["anio", "factor", INC_LAB]:
    persona_geo[col] = pd.to_numeric(persona_geo[col], errors="coerce")
hogar_geo["peso_persona_expandida"] = hogar_geo["factor"] * hogar_geo["tot_integ"]

mapa_keys = mapa_zm[["cve_ent", "cve_mun", "zona_metropolitana"]].drop_duplicates()
hogar_geo = hogar_geo.merge(mapa_keys, on=["cve_ent", "cve_mun"], how="left")
persona_geo = persona_geo.merge(mapa_keys, on=["cve_ent", "cve_mun"], how="left")
hogar_zm = hogar_geo[hogar_geo["zona_metropolitana"].isin(zonas_prioritarias)].copy()
persona_zm = persona_geo[persona_geo["zona_metropolitana"].isin(zonas_prioritarias)].copy()

oficiales = mapa_zm.groupby("zona_metropolitana")["clave_compuesta_municipio"].nunique().to_dict()
coverage_rows = []
for (zona, year), group in hogar_zm.groupby(["zona_metropolitana", "anio"]):
    personas_group = persona_zm[persona_zm["zona_metropolitana"].eq(zona) & persona_zm["anio"].eq(year)]
    coverage_rows.append({"zona_metropolitana": zona, "anio": int(year), "n_hogares": len(group), "n_personas": len(personas_group), "poblacion_expandida": float(personas_group["factor"].sum()), "municipios_oficiales": int(oficiales[zona]), "municipios_presentes_enigh": group[["cve_ent", "cve_mun"]].drop_duplicates().shape[0], "upm_diseno": group[["est_dis", "upm"]].drop_duplicates().shape[0]})
cobertura_zm = pd.DataFrame(coverage_rows).sort_values(["zona_metropolitana", "anio"])
display(cobertura_zm.round(0))


## CDMX: entidad federativa vs Zona Metropolitana del Valle de México

CDMX es una entidad federativa. La Zona Metropolitana del Valle de México incluye CDMX y municipios metropolitanos de México e Hidalgo en la delimitación 2020 usada aquí. No son unidades equivalentes.


In [ ]:
def add_metric_custom(df, label, value_col, weight_col, positive_only=False):
    out = weighted_summary_custom(df, ["anio"], value_col, weight_col, positive_only=positive_only)
    out["metrica"] = label
    return out

cdmx_hogar_geo = hogar_geo[hogar_geo["cve_ent"].eq("09")].copy()
zmvm_hogar = hogar_geo[hogar_geo["zona_metropolitana"].eq("Valle de México")].copy()
cdmx_persona_geo = persona_geo[persona_geo["cve_ent"].eq("09")].copy()

cdmx_metricas_ext = pd.concat([
    add_metric_custom(cdmx_hogar_geo, "CDMX entidad - ingreso corriente hogar", INC_HH, "factor"),
    add_metric_custom(cdmx_hogar_geo, "CDMX entidad - ingreso corriente per cápita", INC_PC, "peso_persona_expandida"),
    add_metric_custom(zmvm_hogar, "ZM Valle de México - ingreso corriente per cápita", INC_PC, "peso_persona_expandida"),
    add_metric_custom(cdmx_persona_geo, "CDMX entidad - ingreso laboral individual positivo", INC_LAB, "factor", positive_only=True),
], ignore_index=True)
display(cdmx_metricas_ext[["anio", "metrica", "n_muestral", "poblacion_expandida", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90", "p95", "gini_ponderado_0_100"]].round(2))

cdmx_plot = cdmx_metricas_ext[cdmx_metricas_ext["anio"].eq(2024) & cdmx_metricas_ext["metrica"].isin(["CDMX entidad - ingreso corriente per cápita", "ZM Valle de México - ingreso corriente per cápita"])].copy()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(cdmx_plot["metrica"], cdmx_plot["mediana_ponderada"], color=["#457b9d", "#e29578"])
ax.set_title("CDMX entidad vs ZM Valle de México, 2024")
ax.set_xlabel("Mediana ponderada, pesos nominales trimestrales")
ax.set_ylabel("Unidad territorial")
ax.text(0.01, -0.22, "Ingreso corriente per cápita del hogar; ponderador poblacional: factor * integrantes.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "cdmx_vs_zmvm_2024.png", "CDMX entidad vs Zona Metropolitana del Valle de México, 2024")


## Comparación entre grandes zonas metropolitanas

Para ingreso corriente per cápita del hogar se usa `factor * tot_integ`, porque la variable representa el ingreso disponible por integrante. Para ingreso laboral individual positivo se usa la base de personas y `factor`.


In [ ]:
met_pc = weighted_summary_custom(hogar_zm, ["anio", "zona_metropolitana"], INC_PC, "peso_persona_expandida")
met_lab = weighted_summary_custom(persona_zm, ["anio", "zona_metropolitana"], INC_LAB, "factor", positive_only=True)

display(Markdown("### Ingreso corriente per cápita del hogar, 2024"))
display(met_pc[met_pc["anio"].eq(2024)][["zona_metropolitana", "n_muestral", "poblacion_expandida", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90", "p95", "gini_ponderado_0_100"]].sort_values("mediana_ponderada", ascending=False).round(2))

display(Markdown("### Ingreso laboral individual positivo, 2024"))
display(met_lab[met_lab["anio"].eq(2024)][["zona_metropolitana", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90"]].sort_values("mediana_ponderada", ascending=False).round(2))


## Figuras territoriales 2024

Estas figuras usan ingreso corriente per cápita del hogar y ponderador poblacional (`factor * tot_integ`). La intención es visualizar brechas territoriales dentro de 2024, no comparar poder adquisitivo entre años.


In [ ]:
regional_pc_doc = weighted_summary_custom(hogar_geo, ["anio", "region_banxico"], INC_PC, "peso_persona_expandida")
tam_loc_doc = weighted_summary_custom(hogar_geo, ["anio", "tam_loc_desc"], INC_PC, "peso_persona_expandida")
est_socio_doc = weighted_summary_custom(hogar_geo, ["anio", "est_socio_desc"], INC_PC, "peso_persona_expandida")

region_2024 = regional_pc_doc[regional_pc_doc["anio"].eq(2024)].copy().sort_values("mediana_ponderada")
fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(region_2024))
ax.hlines(y, region_2024["p25"], region_2024["p75"], color="#7a7a7a", linewidth=5, alpha=0.55, label="P25-P75")
ax.scatter(region_2024["mediana_ponderada"], y, color="#006d77", s=70, label="Mediana ponderada")
ax.set_yticks(y, region_2024["region_banxico"])
ax.set_title("Distribución regional del ingreso corriente per cápita, 2024")
ax.set_xlabel("Pesos nominales trimestrales")
ax.set_ylabel("Región Banxico")
ax.legend(loc="lower right")
ax.text(0.01, -0.16, "Ingreso per cápita del hogar ponderado por población expandida: factor * integrantes.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "ingreso_pc_region_2024.png", "Distribución regional del ingreso corriente per cápita, 2024")

tam_2024 = tam_loc_doc[tam_loc_doc["anio"].eq(2024)].sort_values("mediana_ponderada")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(tam_2024["tam_loc_desc"], tam_2024["mediana_ponderada"], color=["#8d99ae", "#83c5be", "#e9c46a", "#006d77"])
ax.set_title("Gradiente por tamaño de localidad, 2024")
ax.set_xlabel("Mediana ponderada, pesos nominales trimestrales")
ax.set_ylabel("Tamaño de localidad")
ax.text(0.01, -0.20, "Ingreso corriente per cápita del hogar; ponderador poblacional: factor * integrantes.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "gradiente_tam_loc_2024.png", "Gradiente por tamaño de localidad, 2024")

socio_order = ["Bajo", "Medio bajo", "Medio alto", "Alto"]
socio_2024 = est_socio_doc[est_socio_doc["anio"].eq(2024)].copy()
socio_2024["est_socio_desc"] = pd.Categorical(socio_2024["est_socio_desc"], categories=socio_order, ordered=True)
socio_2024 = socio_2024.sort_values("est_socio_desc")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(socio_2024["est_socio_desc"].astype(str), socio_2024["mediana_ponderada"], marker="o", linewidth=2.5, color="#c1121f")
ax.set_title("Gradiente por estrato socioeconómico, 2024")
ax.set_xlabel("Estrato socioeconómico INEGI")
ax.set_ylabel("Mediana ponderada, pesos nominales trimestrales")
ax.text(0.01, -0.18, "Ingreso corriente per cápita del hogar; ponderador poblacional: factor * integrantes.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "gradiente_est_socio_2024.png", "Gradiente por estrato socioeconómico, 2024")

display(Markdown("### Región Banxico, 2024"))
display(region_2024[["region_banxico", "n_muestral", "poblacion_expandida", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90", "gini_ponderado_0_100"]].round(2))


## Grandes urbes vs contexto territorial desfavorecido

No se usa la palabra “marginado” porque aún no se incorporó CONAPO. El grupo desfavorecido se define solo con variables actuales: localidad menor de 2,500 habitantes y `est_socio_desc = Bajo`. Para evitar duplicidades, primero se asignan las tres zonas metropolitanas y luego, fuera de ellas, el grupo desfavorecido.


In [ ]:
hogar_geo["grupo_brecha_territorial"] = pd.NA
mask_metro = hogar_geo["zona_metropolitana"].isin(zonas_prioritarias)
hogar_geo.loc[mask_metro, "grupo_brecha_territorial"] = hogar_geo.loc[mask_metro, "zona_metropolitana"]
mask_desfavorecido = hogar_geo["grupo_brecha_territorial"].isna() & hogar_geo["tam_loc_desc"].eq("Localidades con menos de 2 500 habitantes") & hogar_geo["est_socio_desc"].eq("Bajo")
hogar_geo.loc[mask_desfavorecido, "grupo_brecha_territorial"] = "Localidad pequeña y estrato socioeconómico bajo"

brecha_2024 = weighted_summary_custom(hogar_geo[hogar_geo["anio"].eq(2024) & hogar_geo["grupo_brecha_territorial"].notna()], ["grupo_brecha_territorial"], INC_PC, "peso_persona_expandida")
brecha_2024["p90_p10"] = brecha_2024["p90"] / brecha_2024["p10"]
brecha_2024["p75_p25"] = brecha_2024["p75"] / brecha_2024["p25"]
brecha_2024 = brecha_2024.sort_values("mediana_ponderada", ascending=False)
display(brecha_2024[["grupo_brecha_territorial", "n_muestral", "poblacion_expandida", "media_ponderada", "p10", "p25", "mediana_ponderada", "p75", "p90", "p90_p10", "p75_p25", "gini_ponderado_0_100"]].round(2))

hi = brecha_2024.iloc[0]
lo = brecha_2024.iloc[-1]
display(Markdown(f"**Brecha 2024:** la mediana ponderada de {hi['grupo_brecha_territorial']} es {hi['mediana_ponderada'] / lo['mediana_ponderada']:.2f} veces la de {lo['grupo_brecha_territorial']}."))

fig, ax = plt.subplots(figsize=(9, 5))
plot = brecha_2024.sort_values("mediana_ponderada")
colors = ["#8d99ae" if "Localidad" in group else "#006d77" for group in plot["grupo_brecha_territorial"]]
ax.barh(plot["grupo_brecha_territorial"], plot["mediana_ponderada"], color=colors)
ax.set_title("Brecha territorial: mediana de ingreso per cápita, 2024")
ax.set_xlabel("Mediana ponderada, pesos nominales trimestrales")
ax.set_ylabel("Grupo territorial")
ax.text(0.01, -0.20, "Ingreso corriente per cápita del hogar; ponderador poblacional: factor * integrantes.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "brecha_metropolitana_2024.png", "Brecha territorial por zonas metropolitanas y contexto desfavorecido, 2024")


## Figuras documentales generadas

Estas son las figuras seleccionadas para `reports/documentacion_final_en_desarrollo.md`.


In [ ]:
figuras_documentacion = sorted(p.relative_to(ROOT).as_posix() for p in DOC_FIG_DIR.glob("*.png"))
display(pd.DataFrame({"figura": figuras_documentacion}))


## Resumen de hallazgos actualizado

- `factor` representa un factor de expansión. En `mart_hogar`, `factor` coincide con `factor_hogar`; en 2022 y 2024 las tablas raw que traen `factor` coinciden contra `concentradohogar`. En 2018 y 2020 varias tablas hijas no lo traen, por lo que el mart lo hereda desde hogar.
- El Gini nacional del ingreso corriente total del hogar ponderado por `factor` es 43.83 en 2018, 42.60 en 2020, 41.27 en 2022 y 40.06 en 2024.
- El Gini calculado desde `mart_hogar` y desde `concentradohogar` directo es idéntico hasta cuatro decimales; la construcción del mart no explica la discrepancia con Banxico.
- Frente a Banxico, el cálculo hogar-total queda abajo hasta 3.22 puntos; la variante diagnóstica de ingreso per cápita ponderada por población queda mucho más cerca, con discrepancia máxima cercana a 1.06 puntos.
- Para 2024, Monterrey tiene la mayor mediana ponderada de ingreso corriente per cápita entre las tres zonas metropolitanas revisadas: $24,900. Le siguen Guadalajara ($21,064) y Valle de México ($19,671).
- CDMX entidad no equivale a la Zona Metropolitana del Valle de México: en 2024 la mediana per cápita ponderada de CDMX entidad es $23,930 y la de la ZM Valle de México es $19,671.
- La brecha 2024 entre Monterrey y el grupo “localidad pequeña y estrato socioeconómico bajo” es de 3.16 veces en mediana ponderada de ingreso corriente per cápita.
- Todo sigue en pesos nominales trimestrales. Para comparar niveles monetarios entre años, el siguiente paso es deflactar; para Gini dentro de año, un deflactor común no cambia el índice.
